# Customer Segmentation

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

# --------------------------------------------
# Load Data
# --------------------------------------------
df = banking_marketing_train_encoded.copy()

# --------------------------------------------
# Customer Segmentation
# --------------------------------------------

# Encode Categorical Features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    for col in categorical_cols:
        df[col] = LabelEncoder().fit_transform(df[col])

# Normalize numerical columns using StandardScaler
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
scaler = StandardScaler()
df[numeric_cols] = pd.DataFrame(scaler.fit_transform(df[numeric_cols]),
                                columns=numeric_cols, index=df.index)

# Perform K-Means Clustering (5 Segments)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(df[numeric_cols])

# Define customer segment descriptions
cluster_mapping = {
    0: "Mid-High Income Males with Dependents, Strong Banking Relationship",
    1: "Young, Low-Income Females with Shortest Tenure & Low Credit",
    2: "Older, Low-Income Females with Strong Banking Relationship & High Utilisation",
    3: "Mid-Income Graduates with High Spending & Transactions",
    4: "Educated, Single Individuals with High Credit & Low Utilisation"
}

# Map clusters to descriptions
df['customer_segment'] = df['Cluster'].map(cluster_mapping)

# --------------------------------------------
# Train Models to Predict Customer Segments
# --------------------------------------------

# Select relevant columns for training
selected_cols = [
    'age', 'education', 'default', 'balance', 'housing', 'loan', 'day', 'month', 'duration',
    'campaign', 'pdays', 'previous', 'poutcome_failure', 'poutcome_other', 'poutcome_success',
    'poutcome_unknown', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed',
    'job_services', 'job_student', 'job_technician', 'job_unemployed', 'job_unknown',
    'marital_divorced', 'marital_married', 'marital_single', 'contact_cellular',
    'contact_telephone', 'contact_unknown'
]

df_selected = df[selected_cols].copy()
y = df['Cluster']

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(df_selected, y, test_size=0.2, random_state=42, stratify=y)

# Handle class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Define XGBoost model with class weighting
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', n_estimators=100, learning_rate=0.1, random_state=42)

# Hyperparameter Tuning with GridSearchCV
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'scale_pos_weight': [1, 5]  # Adjust for imbalance
}

grid_search = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
                           param_grid, cv=2, n_jobs=-1, verbose=1)
grid_search.fit(X_train_resampled, y_train_resampled)

# Best parameters from grid search
print(f"Best parameters: {grid_search.best_params_}")

# Use the best model from GridSearchCV
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

# Model evaluation
print(f"\nXGBoost Accuracy with Tuning: {accuracy_score(y_test, y_pred_best):.2f}")
print(classification_report(y_test, y_pred_best, zero_division=1))

# --------------------------------------------
# Ensemble Model (Voting Classifier)
# --------------------------------------------

# Create an ensemble of classifiers
rf = RandomForestClassifier(n_estimators=100, random_state=42)
dt = DecisionTreeClassifier(random_state=42)
ensemble_model = VotingClassifier(estimators=[('xgb', best_model), ('rf', rf), ('dt', dt)], voting='hard')

# Train the ensemble model
ensemble_model.fit(X_train_resampled, y_train_resampled)
y_pred_ensemble = ensemble_model.predict(X_test)

# Evaluate ensemble model
print(f"\nEnsemble Model Accuracy: {accuracy_score(y_test, y_pred_ensemble):.2f}")
print(classification_report(y_test, y_pred_ensemble, zero_division=1))

# --------------------------------------------
# Apply Best Model (XGBoost) to Full Dataset
# --------------------------------------------

df['Predicted_Segment'] = best_model.predict(df_selected)
df['customer_segment'] = df['Predicted_Segment'].map(cluster_mapping)

# Save the entire dataset with segmentation
df.to_csv("data/processed/Q7_banking_marketing_train_segmented.csv", index=False)

print("Customer Segmentation Completed & Full Dataset Saved!")

Fitting 2 folds for each of 16 candidates, totalling 32 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:35:27] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'scale_pos_weight': 1, 'subsample': 0.8}

XGBoost Accuracy with Tuning: 0.87
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2488
           1       0.88      0.96      0.92      1181
           2       0.21      0.44      0.28       198
           3       0.48      0.44      0.46       859
           4       0.95      0.89      0.92      4317

    accuracy                           0.87      9043
   macro avg       0.70      0.74      0.71      9043
weighted avg       0.89      0.87      0.88      9043



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:35:33] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Ensemble Model Accuracy: 0.88
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      2488
           1       0.86      0.97      0.91      1181
           2       0.20      0.28      0.23       198
           3       0.49      0.40      0.44       859
           4       0.94      0.91      0.92      4317

    accuracy                           0.88      9043
   macro avg       0.69      0.71      0.70      9043
weighted avg       0.88      0.88      0.88      9043

Customer Segmentation Completed & Full Dataset Saved!


# Evaluation of Segmentation

1️. Accuracy Overview

*   XGBoost Accuracy with Tuning: 0.87 --> Ensemble Model Accuracy: 0.88
 The ensemble model slightly improves overall performance.



2️. Class-wise Performance (Precision, Recall, F1-score)

*   Observations: Classes 0, 1, and 2 have high precision and recall, which are well-classified. Classes 3 and 4 (minority classes) relatively struggle with lower precison and recall. Ensemble model improves recall for Class 3 but reduces precision for Class 4.

# Multi-armed Bandit (MAB) Model

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import beta

# Load segmented customer dataset
df = pd.read_csv("data/processed/Q7_banking_marketing_train_segmented.csv")

# --------------------------------------------
# Define Banking Campaign Variants
# --------------------------------------------
campaign_variants = [
    "New Account Bonus",
    "Spend-to-Earn Rewards",
    "Free Banking Services",
    "Loyalty Rewards Program"
]
timing_variants = ["Morning", "Afternoon", "Evening"]
channel_variants = ["Email", "SMS", "Push Notification"]

# Initialize parameters for Thompson Sampling (Success/Failure for each campaign)
segment_list = df["customer_segment"].unique()

# Bayesian Priors for Each Segment
historical_performance = df.groupby("customer_segment")["conversion_binary"].mean()
segment_campaign_params = {
    segment: {
        variant: {
            "alpha": max(1, historical_performance[segment] * 10),
            "beta": max(1, (1 - historical_performance[segment]) * 10)
        } for variant in campaign_variants
    }
    for segment in segment_list
}

segment_timing_params = {
    segment: {variant: {"alpha": 1, "beta": 1} for variant in timing_variants}
    for segment in segment_list
}
segment_channel_params = {
    segment: {variant: {"alpha": 1, "beta": 1} for variant in channel_variants}
    for segment in segment_list
}

# --------------------------------------------
# Define Dynamic Fatigue Score Handling
# --------------------------------------------
low_fatigue_threshold = df["fatigue_score"].quantile(0.33)
high_fatigue_threshold = df["fatigue_score"].quantile(0.66)

# --------------------------------------------
# Function to Recommend the Best Campaign for Each Customer
# --------------------------------------------
def recommend_campaign(customer):
    """
    Selects the best campaign, timing, and channel based on:
    - Thompson Sampling with Bayesian Updating
    - Dynamic Fatigue Handling
    - Minimum Subscription Rate Floor
    """

    customer_segment = customer["customer_segment"]
    fatigue_score = customer["fatigue_score"]
    conversion_rate = max(customer["conversion_rate"], 0.01)
    best_contact_time = customer["best_contact_time"]

    # Dynamically categorize fatigue levels
    if fatigue_score <= low_fatigue_threshold:
        fatigue_level = "low"
    elif fatigue_score >= high_fatigue_threshold:
        fatigue_level = "high"
    else:
        fatigue_level = "medium"

    # Adjust campaign selection using fatigue score dynamically
    adjusted_campaigns = campaign_variants.copy()
    if fatigue_level == "high":
        adjusted_campaigns.remove("Spend-to-Earn Rewards")
    elif fatigue_level == "low":
        adjusted_campaigns.append("New Account Bonus")

    # Sample from Beta distributions for campaign selection
    best_campaign = max(
        adjusted_campaigns, key=lambda x: beta.rvs(
            max(segment_campaign_params[customer_segment][x]["alpha"] + conversion_rate * 10, 1),
            max(segment_campaign_params[customer_segment][x]["beta"] + 5, 1)  # Small smoothing term
        )
    )

    # Select best timing based on past engagement
    if best_contact_time in timing_variants:
        best_timing = best_contact_time
    else:
        best_timing = max(
            timing_variants, key=lambda x: beta.rvs(
                max(segment_timing_params[customer_segment][x]["alpha"], 1),
                max(segment_timing_params[customer_segment][x]["beta"], 1)
            )
        )

    # Sample from Beta distributions for channel selection
    best_channel = max(
        channel_variants, key=lambda x: beta.rvs(
            max(segment_channel_params[customer_segment][x]["alpha"], 1),
            max(segment_channel_params[customer_segment][x]["beta"], 1)
        )
    )

    return best_campaign, best_timing, best_channel

# --------------------------------------------
# Generate Recommendations for Each Customer
# --------------------------------------------
df[["recommended_campaign", "recommended_timing", "recommended_channel"]] = df.apply(recommend_campaign, axis=1, result_type="expand")

# Save recommendations
df.to_csv("data/processed/Q7_customer_campaign_recommendations_final.csv", index=False)
print("Final Optimized Banking Campaign Recommendations Generated & Saved! ")


Final Optimized Banking Campaign Recommendations Generated & Saved!


In this question, Multi-Armed Bandit (MAB) algorithm is used to optimize digital marketing campaigns by dynamically adjusting key parameters such as offers, timing, and audience segmentation based on real-time customer interactions.

# Evaluation of MAB Model

In [6]:
# Calculate average subscription rate by segment
segment_subscription_rate = df.groupby('customer_segment')['y'].mean()

print("Average Subscription Rate by Segment:")
print(segment_subscription_rate)

# Check unique values in the target variable y
print(df['y'].unique())

Average Subscription Rate by Segment:
customer_segment
Educated, Single Individuals with High Credit & Low Utilisation                 -0.211618
Mid-High Income Males with Dependents, Strong Banking Relationship              -0.303868
Mid-Income Graduates with High Spending & Transactions                           1.312072
Older, Low-Income Females with Strong Banking Relationship & High Utilisation    1.388273
Young, Low-Income Females with Shortest Tenure & Low Credit                     -0.018168
Name: y, dtype: float64
[-0.36398261  2.74738398]


#Interpretation:

Negative Average Subscription Rates:

Educated, Single Individuals with High Credit & Low Utilisation (-0.21) and Mid-High Income Males with Dependents and Strong Banking Relationship (-0.30) both have negative average subscription rates. This suggests that these segments are less likely to subscribe to the term deposit.

Young, Low-Income Females with Shortest Tenure & Low Credit (-0.02) also has a nearly neutral (slightly negative) rate, indicating a low likelihood of subscription. These customers may not be as engaged with financial products or may have limited trust or interest in term deposits.

Positive Average Subscription Rates:

Mid-Income Graduates with High Spending & Transactions (1.31) and Older, Low-Income Females with Strong Banking Relationship & High Utilisation (1.39) both exhibit positive subscription rates. These segments seem more likely to subscribe, perhaps because they have more consistent interactions with the bank or higher financial literacy and trust in financial products.

The older, low-income females with strong banking relationships and high utilization might have specific trust or long-term engagement with the bank, making them more likely to subscribe to term deposits

There is a significant variation in the subscription rates across different customer segments, suggesting that the factors (income, credit, usage, etc.) affect the likelihood of subscription.

Segments with Positive Subscription Rates are prime targets for marketing campaigns, as they exhibit higher engagement with the term deposit offering.

Segments with Negative Subscription Rates may require further analysis to identify barriers to subscription, such as product mismatch, marketing issues, or other factors that reduce interest.

